# Integração das bases e feriados

Este notebook junta o que foi preparado nos dois anteriores, acrescenta os feriados e persiste a base final em Parquet, com uma linha por dia do Subsistema Sul. A junção é realizada a partir da data, já que as três fontes estão na mesma escala temporal diária e o clima já foi resumido para a região inteira.

In [ ]:
from pathlib import Path

import holidays
import pandas as pd

INTERIM = Path("../data/interim")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

ONS = INTERIM/"ons_diario_sul.parquet"
INMET = INTERIM/"inmet_diario_sul.parquet"
ESTACOES = INTERIM/"inmet_estacoes_sul.parquet"
SAIDA = PROCESSED/"ons_inmet_sul.parquet"

INICIO, FIM = "2021-01-01", "2025-12-31"
UFS = ("PR", "SC", "RS")

### Junção

A base do ONS define as linhas, porque ela já traz um registro para cada dia do período, inclusive os dias sem interrupção. O clima é inetrgado com um JOIN à esquerda.

In [ ]:
interrupcoes = pd.read_parquet(ONS)
clima = pd.read_parquet(INMET)

base = interrupcoes.merge(clima, on="data", how="left", validate="one_to_one")
print(len(base), "dias")

### Feriados

Os feriados vêm da biblioteca holidays, que já traz as datas nacionais e estaduais. Como o subsistema cobre três estados, não existe um único calendário. Criamos uma variável de feriado nacional e outra com a fração dos três estados que estão em feriado naquele dia, que captura os feriados estaduais, como o Dia do Gaúcho. 

Feriados municipais ficam de fora porque não há como associá-los a um único município do subsistema. Também incluímos uma variável para os pontos facultativos, como o Carnaval, que não são feriados legais mas reduzem a atividade de forma parecida.

In [ ]:
anos = range(int(base["data"].dt.year.min()), int(base["data"].dt.year.max()) + 1)
datas = base["data"].dt.date

nacionais = holidays.Brazil(years=anos)
facultativos = holidays.Brazil(years=anos, categories="optional")
estaduais = [holidays.Brazil(years=anos, subdiv=uf) for uf in UFS]

base["feriado_nacional"] = datas.isin(list(nacionais)).astype(int)
base["ponto_facultativo"] = datas.isin(list(facultativos)).astype(int)
base["frac_estados_feriado"] = sum(datas.isin(list(feriados)) for feriados in estaduais) / len(UFS)

base["dia_semana"] = base["data"].dt.dayofweek
base["mes"] = base["data"].dt.month

print("pontos facultativos:")
for data, nome in sorted(facultativos.items()):
    print(" ", data, nome)

print("\nferiados estaduais:")
for uf, calendario_uf in zip(UFS, estaduais):
    for data, nome in sorted((d, n) for d, n in calendario_uf.items() if d not in nacionais):
        print(" ", uf, data, nome)

### Variáveis do dia anterior

O modelo precisa prever um dia usando apenas o que se sabe até o dia anterior. O calendário é conhecido com antecedência, então dia da semana, mês e feriados do próprio dia podem ser usados como estão. Já o clima e as interrupções do dia só são conhecidos depois que o dia acontece, então para a predição usamos as versões deslocadas em um dia, que ganham o sufixo "ontem". As colunas sem sufixo continuam na base porque a segunda análise compara o clima e as interrupções no mesmo dia, mas elas não devem entrar como variáveis explicativas do modelo preditivo. A primeira linha fica sem valores do dia anterior, e isso é tratado na modelagem.

In [ ]:
colunas_do_dia = ["n_perturbacoes", "mw_interrompido", "chuva_media", "chuva_max",
                  "temp_max_media", "temp_min_media", "rajada_max"]

ontem = base[colunas_do_dia].shift(1).add_suffix("_ontem")
base = pd.concat([base, ontem], axis=1)

### Verificação e exportação

Por fim conferimos se há valores ausentes além da primeira linha e exportamos em Parquet. 

In [ ]:
ausentes = base.isna().sum()
print(ausentes[ausentes > 0])

base.to_parquet(f"{PROCESSED}/ons_inmet_sul.parquet", index=False)
print(base.shape)
# base.head()
base.info()